# Enterprise Causal Machine Learning & Uplift Decisioning
## Beyond Predictive Propensity: Maximizing Incremental Campaign ROI via CATE & XGBoost

**Author**: Senior Manager & Lead Data Scientist | Retail Banking & Wealth Analytics  
**Methodology**: Künzel et al. (PNAS 2019) X-Learner with K-Fold Cross-Fitting & Budget-Constrained Argmax Policy Optimization

---

### Executive Problem Statement

Standard marketing response models predict the conditional conversion probability:

$$P(Y=1 \mid X)$$

In retail banking, allocating marketing budget based solely on $P(Y=1 \mid X)$ leads to severe capital misallocation:
1. **Targeting Sure Things**: Customers with high balances who would deposit anyway receive expensive outreach ($15 contact center calls), yielding zero incremental revenue.
2. **Disturbing Sleeping Dogs**: Irritating sensitive customers with high balances who cancel their credit cards or close accounts when contacted, causing massive churn penalties.

**Causal Machine Learning** estimates the **Conditional Average Treatment Effect (CATE)**:

$$\tau(X) = \mathbb{E}[Y^{(1)} - Y^{(0)} \mid X]$$

By targeting customers where $\tau(X) > 0$ and maximizing **Individual Net Expected Value (NEV)**, marketing campaigns routinely achieve **+20% to +40% net dollar lift** on fixed outreach budgets.


In [ ]:
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import xgboost as xgb

# Robust repo root discovery (works from notebook dir or repo root)
curr = Path(os.getcwd()).resolve()
repo_root = None
for parent in [curr] + list(curr.parents):
    if (parent / "scripts" / "generate_synthetic_data.py").exists():
        repo_root = parent
        break

if repo_root is not None:
    for p in [repo_root, repo_root / "scripts", repo_root / "06_projects_finance" / "04_marketing_propensity"]:
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))

from generate_synthetic_data import generate_uplift_dataset
from causal_uplift_engine import (
    XGBoostSLearner,
    XGBoostTLearner,
    XGBoostXLearner,
    compute_qini_curve,
    compute_auuc,
    compute_qini_score,
    bootstrap_permutation_test,
    compute_nev,
    optimize_policy,
    evaluate_policy_realized_profit,
    sweep_budget_frontier,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 120
warnings.filterwarnings("ignore")
print("Environment configured successfully!")


## 1. The Fatal Flaw of Propensity Scoring

Let us simulate a cohort of 5,000 retail banking accounts from an A/B test ($T \in \{0, 1\}$) and train a standard predictive XGBoost model:

$$\hat{P}(Y=1 \mid X)$$

We plot predicted response probability against ground-truth causal uplift $\tau^*(X)$ to reveal the emergent customer archetypes.


In [ ]:
# Generate synthetic RCT banking dataset (15% treatment rate)
df_rct = generate_uplift_dataset(n_samples=5000, seed=42, observational=False)
feature_cols = [
    "account_balance",
    "customer_age",
    "tenure_months",
    "credit_card_spend_30d",
    "web_logins_30d",
    "has_direct_deposit",
]

X = df_rct[feature_cols]
T = df_rct["treatment"].values
Y = df_rct["outcome"].values
tau_star = df_rct["tau_true"].values

# Train standard naive propensity model P(Y=1|X)
clf_prop = xgb.XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.05, random_state=42, eval_metric="logloss")
clf_prop.fit(X, Y)
p_pred = clf_prop.predict_proba(X)[:, 1]

# Visualizing the 4 Customer Archetypes
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(p_pred, tau_star, c=tau_star, cmap="coolwarm", alpha=0.55, edgecolor="none", s=25)
ax.axhline(0, color="black", linestyle="--", linewidth=1.2, alpha=0.7)
ax.axvline(np.median(p_pred), color="gray", linestyle=":", linewidth=1.0, alpha=0.7)

# Archetype Annotations
ax.text(0.70, 0.25, "PERSUADABLES\n(High Uplift: Target Here!)", fontsize=10, fontweight="bold", color="darkgreen", bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="green", alpha=0.8))
ax.text(0.70, -0.18, "SLEEPING DOGS\n(High Conversion, Negative Uplift: Do Not Call!)", fontsize=10, fontweight="bold", color="darkred", bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="red", alpha=0.8))
ax.text(0.85, 0.02, "SURE THINGS\n(Converts Anyway)", fontsize=9, color="blue", bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="blue", alpha=0.6))
ax.text(0.05, 0.02, "LOST CAUSES\n(Never Converts)", fontsize=9, color="gray", bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.6))

ax.set_title("The Propensity Trap: P(Y=1|X) vs True Incremental Uplift τ*(X)", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted Conversion Probability P(Y=1 | X) [Standard Propensity Score]", fontsize=11)
ax.set_ylabel("True Conditional Treatment Effect τ*(X) [Incremental Lift]", fontsize=11)
plt.colorbar(scatter, label="True Uplift τ*")
plt.tight_layout()
plt.show()


## 2. The Foundational Causal Assumption: Unconfoundedness

> ### ⚖️ Rosenbaum & Rubin (1983): Strong Ignorability / Unconfoundedness
>
> $$Y^{(0)}, Y^{(1)} \perp T \mid X$$
>
> **Plain English**: Conditional on observed customer features $X$, treatment assignment $T$ is completely independent of potential outcomes. There are no hidden variables that influence *both* who gets contacted *and* their propensity to convert.
>
> In banking, if relationship managers selectively call high-net-worth customers using private notes or relationship quality not captured in $X$, unconfoundedness is violated.

### Demonstration: Hidden Confounder Bias & Recovery
We simulate an observational campaign where `relationship_manager_quality` ($U$) drives both treatment assignment and baseline conversion.


In [ ]:
# Generate dataset with hidden confounder
df_conf = generate_uplift_dataset(n_samples=5000, seed=42, observational=True, include_hidden_confounder=True)

X_biased = df_conf[feature_cols] # Omit confounder
X_clean = df_conf[feature_cols + ["relationship_manager_quality"]] # Include confounder
T_conf = df_conf["treatment"].values
Y_conf = df_conf["outcome"].values
tau_conf_true = df_conf["tau_true"].values

# Model 1: Naive (Confounded - RM Quality omitted)
xl_biased = XGBoostXLearner(n_estimators=40, max_depth=3, random_state=42).fit(X_biased, T_conf, Y_conf)
tau_biased = xl_biased.predict_cate(X_biased)

# Model 2: Unconfounded (RM Quality included in X)
xl_clean = XGBoostXLearner(n_estimators=40, max_depth=3, random_state=42).fit(X_clean, T_conf, Y_conf)
tau_clean = xl_clean.predict_cate(X_clean)

# Bias Diagnostics
bias_omitted = np.mean(tau_biased - tau_conf_true)
bias_controlled = np.mean(tau_clean - tau_conf_true)
rmse_omitted = np.sqrt(np.mean((tau_biased - tau_conf_true)**2))
rmse_controlled = np.sqrt(np.mean((tau_clean - tau_conf_true)**2))

print("=== IDENTIFICATION ASSUMPTION EMPIRICAL AUDIT ===")
print(f"Omitted Confounder -> Mean Estimation Bias: {bias_omitted:+.4f} | RMSE: {rmse_omitted:.4f}")
print(f"Controlled Confounder -> Mean Estimation Bias: {bias_controlled:+.4f} | RMSE: {rmse_controlled:.4f}")
print(f"Bias Reduction from Controlling Confounder: {abs(bias_omitted - bias_controlled)/abs(bias_omitted)*100:.1f}%")


## 3. Meta-Learner Comparison: S-Learner vs T-Learner vs X-Learner

We now benchmark the three canonical meta-learning architectures on our RCT cohort:

1. **S-Learner (Single Model)**: Fits $\mu(X, T)$. Vulnerable to tree regularization dropping $T$ when $|X| \gg 1$.
2. **T-Learner (Two Models)**: Fits $\mu_1(X)$ and $\mu_0(X)$ separately. Vulnerable to high variance when $N_T \ll N_C$.
3. **X-Learner (Künzel et al. 2019)**: Imputes counterfactual residuals with K-fold cross-fitting and combines via propensity weighting:
   $$\hat{\tau}_X(x) = e(x)\hat{\tau}_0(x) + (1 - e(x))\hat{\tau}_1(x)$$


In [ ]:
# 1. Fit S-Learner
s_learner = XGBoostSLearner(n_estimators=40, max_depth=3, random_state=42).fit(X, T, Y)
tau_s = s_learner.predict_cate(X)
diag_s = s_learner.get_treatment_split_diagnostics()

# 2. Fit T-Learner
t_learner = XGBoostTLearner(n_estimators=40, max_depth=3, random_state=42).fit(X, T, Y)
tau_t = t_learner.predict_cate(X)

# 3. Fit X-Learner (Cross-Fitted)
x_learner = XGBoostXLearner(n_estimators=40, max_depth=3, n_splits=5, use_cross_fitting=True, random_state=42).fit(X, T, Y)
tau_x = x_learner.predict_cate(X)

print(f"S-Learner Split Diagnostic: Treatment feature selected in only {diag_s['treatment_split_fraction']*100:.1f}% of splits!")
print(f"T-Learner vs X-Learner Correlation with Ground Truth τ*: T={np.corrcoef(tau_t, tau_star)[0,1]:.3f} vs X={np.corrcoef(tau_x, tau_star)[0,1]:.3f}")


In [ ]:
# Compute Qini Curves
n_eval_bins = 20
fracs, q_x = compute_qini_curve(Y, T, tau_x, n_bins=n_eval_bins)
_, q_t = compute_qini_curve(Y, T, tau_t, n_bins=n_eval_bins)
_, q_s = compute_qini_curve(Y, T, tau_s, n_bins=n_eval_bins)
_, q_oracle = compute_qini_curve(Y, T, tau_star, n_bins=n_eval_bins)
_, q_rand = compute_qini_curve(Y, T, np.random.RandomState(42).randn(len(Y)), n_bins=n_eval_bins)

# Compute AUUC and Permutation Significance
auuc_x = compute_auuc(Y, T, tau_x)
auuc_t = compute_auuc(Y, T, tau_t)
auuc_s = compute_auuc(Y, T, tau_s)
auuc_oracle = compute_auuc(Y, T, tau_star)
perm_test_x = bootstrap_permutation_test(Y, T, tau_x, n_bootstrap=300, random_state=42)

# Plot Qini Curves
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(fracs, q_oracle, label=f"Oracle τ* (AUUC={auuc_oracle:.4f})", color="black", linestyle="--", linewidth=2.0)
ax.plot(fracs, q_x, label=f"X-Learner (AUUC={auuc_x:.4f}, p={perm_test_x['p_value']:.3f})", color="#1f77b4", linewidth=2.2)
ax.plot(fracs, q_t, label=f"T-Learner (AUUC={auuc_t:.4f})", color="#ff7f0e", linewidth=1.8)
ax.plot(fracs, q_s, label=f"S-Learner (AUUC={auuc_s:.4f})", color="#2ca02c", linewidth=1.5)
ax.plot(fracs, q_rand, label="Random Baseline", color="gray", linestyle=":", linewidth=1.2)

ax.set_title("Cumulative Qini Curves: Meta-Learner Benchmark", fontsize=13, fontweight="bold")
ax.set_xlabel("Targeted Population Fraction", fontsize=11)
ax.set_ylabel("Cumulative Incremental Conversions Q(k)", fontsize=11)
ax.legend(frameon=True, facecolor="white", loc="lower right")
plt.tight_layout()
plt.show()

# Summary Metric Table
pd.DataFrame({
    "Meta-Learner": ["X-Learner (Cross-Fitted)", "T-Learner", "S-Learner", "Random Targeting"],
    "AUUC": [auuc_x, auuc_t, auuc_s, compute_auuc(Y, T, np.zeros(len(Y)))],
    "Qini Score (vs Oracle)": [auuc_x/auuc_oracle, auuc_t/auuc_oracle, auuc_s/auuc_oracle, 0.0],
    "Permutation p-value": [perm_test_x["p_value"], "N/A", "N/A", 1.0],
    "Statistically Significant": [perm_test_x["is_significant"], "N/A", "N/A", False],
})


## 4. Cross-Fitting Diagnostic: Mitigating In-Sample Residual Bias

Stage 2 of the X-Learner models imputed counterfactual differences:
$D_1 = Y_1 - \hat{\mu}_0(X_1)$ and $D_0 = \hat{\mu}_1(X_0) - Y_0$.

If $\hat{\mu}_0$ and $\hat{\mu}_1$ are evaluated in-sample on the same training units, the residual variance is artificially deflated, inducing in-sample optimism. We verify how 5-fold cross-fitting eliminates this bias.


In [ ]:
# Compare X-Learner with and without Cross-Fitting
xl_nocf = XGBoostXLearner(n_estimators=40, max_depth=3, use_cross_fitting=False, random_state=42).fit(X, T, Y)
tau_nocf = xl_nocf.predict_cate(X)

rmse_cf = np.sqrt(np.mean((tau_x - tau_star)**2))
rmse_nocf = np.sqrt(np.mean((tau_nocf - tau_star)**2))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(tau_x - tau_star, bins=40, alpha=0.6, label=f"With 5-Fold Cross-Fitting (RMSE={rmse_cf:.4f})", color="steelblue", density=True)
ax.hist(tau_nocf - tau_star, bins=40, alpha=0.5, label=f"Without Cross-Fitting (RMSE={rmse_nocf:.4f})", color="darkorange", density=True)
ax.axvline(0, color="black", linestyle="--")
ax.set_title("Error Distribution (τ̂ - τ*): Cross-Fitting vs In-Sample Imputation", fontsize=12, fontweight="bold")
ax.set_xlabel("Estimation Error", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.legend(frameon=True, facecolor="white")
plt.tight_layout()
plt.show()


## 5. Model Explainability via TreeSHAP on $\hat{\tau}(X)$

### Regulatory & Governance Warning (SR 11-7 / ECOA)

> [!WARNING]
> **SR 11-7 & ECOA Compliance Notice**:  
> SHAP values computed on the second-stage uplift regressor $\hat{\tau}_1(X)$ are **model-feature attribution scores**, NOT causal attributions.
> - They explain how features alter the *model's predicted uplift*, not the true causal mechanism in the real world.
> - Under collinearity, TreeSHAP arbitrarily splits attribution across correlated features (e.g. `account_balance` and `credit_card_spend_30d`).
> - **Regulatory mandate**: These SHAP values must **never** be cited in adverse action notices or regulatory audits as proof of causal impact.


In [ ]:
# Compute TreeSHAP on second-stage uplift model tau_1
explainer = shap.TreeExplainer(x_learner.model_tau1)
shap_values = explainer(X)

plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values, max_display=7, show=False)
plt.title("TreeSHAP on Uplift Regressor τ̂₁(X) (Model Attribution, Not Causal Proof)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Budget-Constrained Argmax Policy Optimizer

### Closed-Form Policy Rule
For outreach cost $C_{\text{contact}} = \$15$, conversion reward $V_{\text{convert}} = \$250$, and churn penalty $P_{\text{churn}} = \$300$:

$$\text{NEV}_i = \max(\hat{\tau}_i, 0) \cdot V_{\text{convert}} - C_{\text{contact}} - \max(-\hat{\tau}_i, 0) \cdot P_{\text{churn}}$$

$$\mathcal{S}^* = \{ i : \text{NEV}_i > 0 \} \cap \text{Top-}B \text{ ranked by } \text{NEV}_i$$

We sweep outreach budget from 2% to 40% of the customer base and compare realized net campaign profit.


In [ ]:
# Sweep Budget Frontier
budget_df = sweep_budget_frontier(
    tau_hat=tau_x,
    naive_propensity_scores=p_pred,
    tau_true=tau_star,
    budget_grid=np.linspace(0.02, 0.40, 20),
    v_convert=250.0,
    c_contact=15.0,
    p_churn=300.0,
)

# Plot Budget Frontier
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(budget_df["budget_ratio"]*100, budget_df["profit_causal"]/1000, marker="o", label="Strategy A: Causal Uplift (NEV Argmax)", color="#1f77b4", linewidth=2.2)
ax.plot(budget_df["budget_ratio"]*100, budget_df["profit_naive"]/1000, marker="s", label="Strategy B: Naive Propensity P(Y=1|X)", color="#d62728", linewidth=2.0)
ax.plot(budget_df["budget_ratio"]*100, budget_df["profit_random"]/1000, marker="^", label="Strategy C: Random Dialing Baseline", color="gray", linestyle="--", linewidth=1.5)

ax.axvline(15, color="black", linestyle=":", label="Standard 15% Budget Threshold")
ax.set_title("Campaign Profit Frontier: Causal Uplift vs Predictive Propensity", fontsize=13, fontweight="bold")
ax.set_xlabel("Marketing Budget (% of Customer Base Contacted)", fontsize=11)
ax.set_ylabel("Realized Net Campaign Profit ($k)", fontsize=11)
ax.legend(frameon=True, facecolor="white", loc="lower right")
plt.tight_layout()
plt.show()


## 7. Executive Summary & CMO Business Case

### Financial Performance at 15% Outreach Budget (750 Customers Contacted)


In [ ]:
# Evaluate 15% budget performance
b = 0.15
mask_causal = optimize_policy(tau_x, budget_ratio=b)
res_causal = evaluate_policy_realized_profit(mask_causal, Y, T, tau_true=tau_star)

naive_rank = np.argsort(-p_pred)
mask_naive = np.zeros(len(Y), dtype=bool)
mask_naive[naive_rank[:int(b * len(Y))]] = True
res_naive = evaluate_policy_realized_profit(mask_naive, Y, T, tau_true=tau_star)

perf_summary = pd.DataFrame([
    {
        "Targeting Strategy": "Naive Propensity P(Y=1|X)",
        "Accounts Contacted": res_naive["n_contacted"],
        "Campaign Cost ($)": f"${res_naive['campaign_cost']:,.0f}",
        "Gross Incremental Revenue ($)": f"${res_naive['gross_revenue']:,.0f}",
        "Sleeping Dog Churn Penalty ($)": f"${res_naive['churn_penalty']:,.0f}",
        "Net Campaign Profit ($)": f"${res_naive['net_profit']:,.0f}",
        "Profit Lift over Naive": "Baseline (0%)"
    },
    {
        "Targeting Strategy": "Causal Uplift (NEV Argmax)",
        "Accounts Contacted": res_causal["n_contacted"],
        "Campaign Cost ($)": f"${res_causal['campaign_cost']:,.0f}",
        "Gross Incremental Revenue ($)": f"${res_causal['gross_revenue']:,.0f}",
        "Sleeping Dog Churn Penalty ($)": f"${res_causal['churn_penalty']:,.0f}",
        "Net Campaign Profit ($)": f"${res_causal['net_profit']:,.0f}",
        "Profit Lift over Naive": f"+{(res_causal['net_profit'] - res_naive['net_profit'])/abs(res_naive['net_profit'])*100:.1f}%"
    }
])

display(perf_summary)
print(f"\n🎯 Key Financial Takeaway: Switching from predictive propensity to causal uplift delivers ${res_causal['net_profit'] - res_naive['net_profit']:,.2f} in pure net profit on a single campaign!")
